# Sensitivity Sweep (Bali example)

Standalone notebook for the LSEQM+DL parameter sensitivity analysis (thesis Section 4.8). It re-runs the correction and metric pipeline over the Bali subdomain while varying, one at a time:

- `DL_BLEND_ALPHA` (CNN blend weight, default 0.70)
- `GPD_THRESHOLD_PERCENTILE` (default 80)
- `DENSITY_SATURATION_COUNT` (default 2)

For each setting it pools the per-pixel CPC-validated metrics over land pixels and the chosen dekads, prints a comparison table, and saves `sensitivity_sweep_bali.csv` to the Bali output directory.

**Run order:** run every cell top to bottom. On Colab, the first cell mounts Google Drive; locally, skip it and set `ROOT` in the setup cell.

## 1 Connect Google Drive (Colab only)

In [ ]:
from google.colab import drive
import os

if os.path.exists('/content/drive'):
    try:
        drive.flush_and_unmount()
        print('Successfully unmounted')
    except Exception:
        print('Unmount failed, the drive might not be mounted or busy')

drive.mount('/content/drive')

## 2 Install packages (only if needed)

In [ ]:
# In Google Colab almost all packages are already available, except netCDF4
!pip install netCDF4

## 3 Setup environment and load configuration

In [ ]:
import sys, os, importlib, logging
logging.basicConfig(level=logging.INFO)

# Windows DLL fix for local conda (no-op on Colab)
if sys.platform == 'win32':
    _p = os.environ.get('CONDA_PREFIX') or sys.prefix
    for _d in [os.path.join(_p,'Library','bin'), os.path.join(_p,'Library','lib'),
               os.path.join(_p,'Library','mingw-w64','bin'), os.path.join(_p,'bin'), _p]:
        if os.path.isdir(_d):
            try: os.add_dll_directory(_d)
            except OSError: pass
            if _d not in os.environ.get('PATH',''):
                os.environ['PATH'] = _d + os.pathsep + os.environ.get('PATH','')

# Project root: Colab Drive path by default; change for local Jupyter.
ROOT = '/content/drive/MyDrive/hybrid-bias-correction'
# ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))   # local
assert os.path.isfile(os.path.join(ROOT,'src','config.py')), f'Not found: {ROOT}'

if ROOT not in sys.path:
    sys.path.insert(0, ROOT)
for m in [k for k in sys.modules if k.startswith('src')]:
    del sys.modules[m]

import src.config as _cfg
importlib.reload(_cfg)
CONFIG_FILE = 'config_bali.yml'
_cfg.initialize_config(os.path.join(ROOT, CONFIG_FILE))

from src import config
print('Config loaded:')
print('  output_dir :', config.output_dir)
print('  IMERGL     :', config.imergl_file)
print('  CPC        :', config.cpc_file)
print('  alpha      :', config.DL_BLEND_ALPHA)
print('  gpd pctl   :', config.GPD_THRESHOLD_PERCENTILE)
print('  saturation :', config.DENSITY_SATURATION_COUNT)

## 4 Load datasets (IMERG-L, CPC, native CPC) + land-sea mask

In [ ]:
import xarray as xr
from src import config
from src.utility import apply_land_sea_mask

_engine = config.NETCDF_ENGINE
imerg_ds = xr.open_dataset(config.imergl_file, decode_times=True, engine=_engine)
cpc_ds   = xr.open_dataset(config.cpc_file,   decode_times=True, engine=_engine)

if hasattr(config,'cpc_native_file') and os.path.isfile(config.cpc_native_file):
    cpc_ds_native = xr.open_dataset(config.cpc_native_file, decode_times=True, engine=_engine)
else:
    cpc_ds_native = None
    print('WARNING: native 0.5deg CPC not found; block artefacts may appear')

imerg_ds[config.IMERG_PRECIP_VAR] = apply_land_sea_mask(
    imerg_ds[config.IMERG_PRECIP_VAR], config.mask_file)
cpc_ds[config.CPC_PRECIP_VAR] = apply_land_sea_mask(
    cpc_ds[config.CPC_PRECIP_VAR], config.mask_file)
print('Datasets loaded and land-sea mask applied.')

## 5 Align CPC to the IMERG grid

In [ ]:
from src.utility import load_mask, reindex_and_align_with_monotonicity

land_sea_mask = load_mask(config.mask_file)
cpc_ds_aligned, _ = reindex_and_align_with_monotonicity(imerg_ds, cpc_ds, land_sea_mask)
print('Alignment complete: imerg_ds, cpc_ds_aligned, cpc_ds_native are ready.')

## 6 Run the sensitivity sweep

Edit `DEKADS` to control coverage. The default is four representative dekads (wet / shoulder / dry) so a five-point sweep stays tractable on free Colab. For the full record set `DEKADS = [(m, d) for m in range(1, 13) for d in (1, 2, 3)]` (much longer).

In [ ]:
# Sensitivity sweep over the Bali subdomain.
# Reuses imerg_ds, cpc_ds_aligned, cpc_ds_native defined above.
# set_param() broadcasts each override into every module that binds the
# name (some modules do `from config import X`), so no kernel restart is
# needed between settings.

import importlib
import inspect
import glob
import os
import numpy as np
import pandas as pd
import xarray as xr

import src.config as config
import src.utility as _util
import src.station_density as _sd
from src.bias_correction import run_correction_pipeline
from src.metrics import run_metrics_pipeline

# ------------------------------------------------------------------
# Sweep configuration - edit here
# ------------------------------------------------------------------
# All 36 dekadal windows (months 1-12, dekads 1-3). This is the full run.
# Each setting re-runs correction + metrics over all 36 dekads, so the full
# 3-parameter x 5-value sweep is long (~10-15 h); run it on Colab Pro.
# For a quick first look, switch to the 4-dekad subset on the line below.
DEKADS = [(m, d) for m in range(1, 13) for d in (1, 2, 3)]
# DEKADS = [(1, 2), (4, 2), (7, 2), (10, 2)]   # quick subset: wet/shoulder/dry

SWEEPS = {
    "alpha": {"param": "DL_BLEND_ALPHA",          "values": [0.5, 0.6, 0.7, 0.8, 0.9], "default": 0.70},
    "gpd":   {"param": "GPD_THRESHOLD_PERCENTILE", "values": [70, 75, 80, 85, 90],       "default": 80},
    "csat":  {"param": "DENSITY_SATURATION_COUNT", "values": [1, 2, 3, 4, 5],            "default": 2},
}

# Per-pixel metric variables to pool from the metrics NetCDF (only those
# present are used; names follow src/metrics.py output).
POOL_VARS = ["pearson_correlation", "relative_bias", "stdev_ratio",
             "rmse", "nse"]

METHOD = "lseqmdl"

# Modules that may hold a local binding of a swept parameter.
_SRC_MODULES = [
    "src.config", "src.distribution_fitting", "src.deep_learning",
    "src.bias_correction", "src.station_density", "src.metrics",
]


def set_param(name, value):
    """Broadcast a config override into every module that binds it.

    Two traps handled here:
    (1) Modules that do `from .config import X` hold their own binding, so we
        patch the name in every module that has it (not just config).
    (2) apply_deeplearning_model() captures blend_alpha as a DEFAULT ARGUMENT
        frozen at import time, and bias_correction calls it WITHOUT passing
        blend_alpha. Patching the module global is therefore not enough for
        DL_BLEND_ALPHA - we must rewrite the function's __defaults__ too, or the
        alpha sweep is a silent no-op.
    """
    n = 0
    for modname in _SRC_MODULES:
        mod = importlib.import_module(modname)
        if hasattr(mod, name):
            setattr(mod, name, value)
            n += 1

    extra = ""
    if name == "DL_BLEND_ALPHA":
        dl = importlib.import_module("src.deep_learning")
        f = dl.apply_deeplearning_model
        sig = inspect.signature(f)
        new_defaults = tuple(
            value if pn == "blend_alpha" else p.default
            for pn, p in sig.parameters.items()
            if p.default is not inspect.Parameter.empty
        )
        f.__defaults__ = new_defaults
        eff = inspect.signature(f).parameters["blend_alpha"].default
        extra = f"; apply_deeplearning_model blend_alpha default -> {eff}"

    print(f"    set {name} = {value}  (patched in {n} module(s)){extra}")


# Metric NetCDFs use the start-day-of-dekad token in their filename:
#   dekad index 1 -> "dekad01", 2 -> "dekad11", 3 -> "dekad21".
# Globbing on the bare index would mis-match (e.g. "dekad1" hits "dekad11").
DEKAD_TOKEN = {1: "01", 2: "11", 3: "21"}

# ------------------------------------------------------------------
# Forcing regeneration - WHY this is necessary
# ------------------------------------------------------------------
# The pipeline defaults to existing_file_action="skip" and, in non-interactive
# mode, HARDCODES model reuse ('U'). It also caches the file-skip decision and
# the confidence mask in memory. If left alone, every sweep setting after the
# first would silently reuse stale outputs and report identical numbers.
# We therefore (a) clear the relevant caches and (b) delete the outputs for the
# dekads being recomputed, so each setting genuinely re-runs from scratch.
_PURGE_SUBDIRS = [
    "corrected_ls", "corrected_lseqm", "corrected_lseqmdl",
    "metrics_ls", "metrics_lseqm", "metrics_lseqmdl",
    "quality_ls", "quality_lseqm", "quality_lseqmdl",
]


def reset_for_new_setting():
    """Clear caches and force overwrite before a new parameter setting."""
    _util.reset_user_choice()                 # clear cached file-skip decision
    config.EXISTING_FILE_ACTION = "overwrite"  # belt-and-suspenders
    _sd._confidence_cache.clear()             # drop cached confidence mask
    # delete the confidence-mask file so it regenerates with the current csat
    cmf = getattr(config, "CONFIDENCE_MASK_FILE", None)
    if cmf and os.path.isfile(cmf):
        try:
            os.remove(cmf)
        except OSError:
            pass


def purge_dekad_outputs(m, d):
    """Delete corrected/metrics/quality/model files for one dekad so they regenerate."""
    tok = DEKAD_TOKEN[d]
    pat = f"*month{m:02d}_dekad{tok}*"
    for sub in _PURGE_SUBDIRS:
        for f in glob.glob(f"{config.output_dir}/{sub}/{pat}"):
            try:
                os.remove(f)
            except OSError:
                pass
    # trained CNN for this dekad (forces retrain - required for the gpd sweep)
    mdl = (f"{config.output_dir}/trained_models/"
           f"bias_correction_model_month{m:02d}_dekad{tok}.keras")
    if os.path.isfile(mdl):
        try:
            os.remove(mdl)
        except OSError:
            pass


def pool_metrics(dekads):
    """Pool land-pixel medians from the CPC daily-timeseries metric file.

    The metric pipeline writes SIX files per dekad: the daily-timeseries ("ts")
    and single-dekad ("sd") metrics, each against three references
    (cpc, imergl, imergf). Only the "metricsts_cpc" file (daily-paired against
    CPC) is the thesis basis for Pearson r and the headline metrics; the
    "imergl"/"imergf" files compare the corrected product against the satellite
    it was derived from and so carry near-unity correlation that would corrupt
    the pool. We therefore anchor strictly on metricsts_cpc.
    """
    rows = {v: [] for v in POOL_VARS}
    for (m, d) in dekads:
        tok = DEKAD_TOKEN[d]
        patt = (f"{config.output_dir}/metrics_{METHOD}/"
                f"*metricsts_cpc_imergl_{METHOD}_month{m:02d}_dekad{tok}*.nc4")
        files = sorted(glob.glob(patt))
        if not files:
            print(f"    WARNING: no metricsts_cpc file for month{m:02d} dekad{tok}")
        for f in files:
            ds = xr.open_dataset(f, decode_timedelta=False)
            for v in POOL_VARS:
                if v in ds:
                    a = ds[v].values.astype(float).ravel()
                    rows[v].extend(a[~np.isnan(a)].tolist())
            ds.close()
    return {v: (float(np.median(rows[v])) if rows[v] else np.nan) for v in POOL_VARS}


def run_setting(dekads):
    """Run correction + metrics for the dekads at the current config, return pooled medians."""
    reset_for_new_setting()
    for (m, d) in dekads:
        purge_dekad_outputs(m, d)
        run_correction_pipeline(imerg_ds, cpc_ds_aligned, m, d,      # noqa: F821
                                cpc_native_ds=cpc_ds_native)         # noqa: F821
        run_metrics_pipeline(m, d, mode="timeseries")
    return pool_metrics(dekads)


# ------------------------------------------------------------------
# Run the sweeps
# ------------------------------------------------------------------
# Checkpoint the CSV after every setting so a long run that is
# interrupted still leaves all completed rows on disk.
out_csv = f"{config.output_dir}/sensitivity_sweep_bali.csv"

results = []
for sweep_name, spec in SWEEPS.items():
    param = spec["param"]
    print(f"\n===== sweep: {sweep_name} ({param}) =====")
    for val in spec["values"]:
        print(f"  [{sweep_name} = {val}]")
        # reset all three to defaults, then set the swept one
        for s in SWEEPS.values():
            set_param(s["param"], s["default"])
        set_param(param, val)
        med = run_setting(DEKADS)
        row = {"sweep": sweep_name, "param": param, "value": val}
        row.update(med)
        results.append(row)
        # write a checkpoint after each setting
        pd.DataFrame(results).to_csv(out_csv, index=False)
        print("    ->", {k: round(v, 3) for k, v in med.items() if not np.isnan(v)},
              f"(checkpointed {len(results)} row(s) -> {out_csv})")

# restore defaults
for s in SWEEPS.values():
    set_param(s["param"], s["default"])

df = pd.DataFrame(results)
df.to_csv(out_csv, index=False)
print(f"\nSaved: {out_csv}")
print(df.to_string(index=False))


## 7 Inspect results

The table above and `sensitivity_sweep_bali.csv` give the pooled median metric per parameter value. Send the CSV back for the thesis Section 4.8 table.